# 4주차 학생용 실습 — 결측값과 이상한 데이터 정리하기

오늘의 질문: **빈 칸이 있거나 말이 안 되는 값이 있으면 분석 결과가 어떻게 달라질까?**

이 노트북은 위에서 아래로 순서대로 실행합니다. `# TODO` 라고 적힌 부분만 직접 채워 넣으면 됩니다.

## 1단계. Pandas 불러오고 데이터 읽기

In [ ]:
import pandas as pd
df = pd.read_csv("../../data/weekly/week04/week04_dirty_process_data.csv")
df.head()

## 2단계. 데이터 크기 확인하기
`shape`로 (행 개수, 열 개수)를 확인합니다.

In [ ]:
print(df.shape)

## 3단계. 결측값 개수 세기
각 칸이 결측값인지 알려주는 메서드는 `isna()`입니다. 여기에 `.sum()`을 붙이면
열별 결측값 개수를 셀 수 있습니다.

In [ ]:
# TODO: 빈칸에 'isna'를 입력해 열별 결측값 개수를 확인하세요.
print(df.____().sum())

**질문**: 어떤 열에 결측값이 가장 많은가요?

> ___ 열에 결측값이 ___개로 가장 많다.

## 4단계. 결측값이 있는 행 직접 보기
`df.isna().any(axis=1)`은 '이 행에 결측값이 하나라도 있는가'를 알려줍니다.
이 조건을 대괄호 `[ ]` 안에 넣으면 그 행들만 골라낼 수 있습니다.

In [ ]:
# TODO: 빈칸에 'isna'를 입력해 결측값이 있는 행만 확인하세요.
df[df.____().any(axis=1)]

## 5단계. 결측값 제거 vs 대체 비교
`dropna()`는 결측이 있는 행을 제거하고, `fillna(값)`은 결측 칸을 값으로 채웁니다.
온도처럼 이상값이 섞여 있을 수 있는 열은 평균보다 **중앙값(median)**으로 채우는 것이 더 안전합니다.

In [ ]:
# 결측이 있는 행을 제거해봅니다.
df_dropped = df.dropna()
print("제거 후 행 개수:", df_dropped.shape[0])

In [ ]:
# TODO: 빈칸에 'median'을 입력해 온도_섭씨의 중앙값을 구하세요.
median_temp = df["온도_섭씨"].____()
print("온도_섭씨 중앙값:", median_temp)

**질문**: 제거(`dropna`)와 대체(`fillna`) 중 어떤 방법이 행 개수를 더 많이 유지하나요?

> ___ 방법이 행 개수를 더 많이 유지한다.

## 6단계. 중복 행 찾고 정리하기
`duplicated()`는 이전에 나온 행과 완전히 같은 행이면 `True`를 돌려줍니다.

In [ ]:
# TODO: 빈칸에 'duplicated'를 입력해 중복 행 개수를 확인하세요.
print(df.____().sum())

df_no_dup = df.drop_duplicates()
print("중복 제거 후 행 개수:", df_no_dup.shape[0])

## 7단계. 이상값 찾기
정상 범위(온도 295~305℃, 압력 995~1025 Pa)를 크게 벗어난 값을 조건 검색으로 찾아봅니다.

In [ ]:
# 음수 압력을 가진 행을 찾습니다.
df[df["압력_Pa"] < 0]

In [ ]:
# TODO: 부등호를 채워 온도가 400℃보다 큰(극단 고온) 행을 찾으세요.
df[df["온도_섭씨"] ___ 400]

## 8단계(도전). 정제 전후 비교하기
지금까지 배운 것을 모두 합쳐 데이터를 정제한 뒤, 정제 전/후 행 개수와 평균 온도를 비교해봅니다.

In [ ]:
# 1) 중복 제거
df_clean = df.drop_duplicates().copy()

# 2) 이상값(극단 고온, 음수 압력, 비정상 가스유량) 제거
bad_temp = df_clean["온도_섭씨"] > 400
bad_pressure = df_clean["압력_Pa"] < 0
bad_gas = (df_clean["가스유량_slm"] < 0) | (df_clean["가스유량_slm"] > 200)
df_clean = df_clean[~(bad_temp | bad_pressure | bad_gas)].copy()

# 3) 합격여부가 결측인 행 제거(범주형 값은 대체가 어려움)
df_clean = df_clean.dropna(subset=["합격여부"])

# TODO: 빈칸에 'fillna'를 입력해 남은 결측값을 중앙값으로 채우세요.
df_clean = df_clean.____({
    "온도_섭씨": df_clean["온도_섭씨"].median(),
    "압력_Pa": df_clean["압력_Pa"].median(),
    "가스유량_slm": df_clean["가스유량_slm"].median(),
})

print("정제 후 행 개수:", df_clean.shape[0])
print("정제 후 결측값 합계:", df_clean.isna().sum().sum())

In [ ]:
mean_before = df["온도_섭씨"].mean()
mean_after = df_clean["온도_섭씨"].mean()
print("정제 전 평균 온도:", round(mean_before, 2))
print("정제 후 평균 온도:", round(mean_after, 2))
print("행 개수:", df.shape[0], "->", df_clean.shape[0])

**질문**: 정제 전과 후의 평균 온도는 몇 ℃ 차이가 나나요? 왜 그런 차이가 생겼을까요?

> 정제 전 평균 ___℃, 정제 후 평균 ___℃, 차이는 약 ___℃이다. 그 이유는 ___이다.

## 9단계. 오늘의 분석을 한 문장으로 정리하기

아래 마크다운 셀을 더블클릭해서 여러분의 문장으로 바꿔보세요.

> (예시) 원본 데이터 113행 중 중복 3건, 이상값 6건, 결측값이 있는 행을 정리한 결과 최종 102행이 남았으며, 정제 전후 평균 온도는 약 4.2℃ 차이가 났다.